In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.chdir('/content/drive/MyDrive/UNSW_Blanced')

In [4]:
!pip install adversarial-robustness-toolbox


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.9 MB/s eta 0:00:00


In [5]:
#@title Env & setup
!pip -q install numpy pandas scikit-learn scipy

import os, time, random, math, json
import numpy as np
import pandas as pd
from math import ceil
from typing import Dict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score

from scipy.stats import norm
from scipy.stats import beta as beta_dist


def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [6]:


#@title Load UNSW dataset → tensors
import pandas as pd
import numpy as np
import torch

train = pd.read_csv('unsw_multi_train_processed.csv')
test = pd.read_csv('unsw_multi_test_processed.csv')

X_train = train.drop(columns=['Label']).values
y_train = train['Label'].values
X_test  = test.drop(columns=['Label']).values
y_test  = test['Label'].values

X_train_tensor = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train_tensor = torch.tensor(np.array(y_train), dtype=torch.long)
X_test_tensor  = torch.tensor(np.array(X_test),  dtype=torch.float32)
y_test_tensor  = torch.tensor(np.array(y_test),  dtype=torch.long)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Test :", X_test_tensor.shape,  y_test_tensor.shape)




Train: torch.Size([175341, 194]) torch.Size([175341])
Test : torch.Size([82332, 194]) torch.Size([82332])


In [7]:


#@title Load UNSW dataset → tensors
import pandas as pd
import numpy as np
import torch

train = pd.read_csv('unsw_multi_train_processed.csv')
test = pd.read_csv('unsw_multi_test_processed.csv')

X_train = train.drop(columns=['Label']).values
y_train = train['Label'].values
X_test  = test.drop(columns=['Label']).values
y_test  = test['Label'].values

X_train_tensor = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train_tensor = torch.tensor(np.array(y_train), dtype=torch.long)
X_test_tensor  = torch.tensor(np.array(X_test),  dtype=torch.float32)
y_test_tensor  = torch.tensor(np.array(y_test),  dtype=torch.long)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Test :", X_test_tensor.shape,  y_test_tensor.shape)




Train: torch.Size([175341, 194]) torch.Size([175341])
Test : torch.Size([82332, 194]) torch.Size([82332])


In [8]:
#@title Base model
import torch.nn as nn

class TabularNN(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        return self.fc4(x)


In [9]:
num_classes = 10
model = TabularNN(input_size=X_train_tensor.shape[1], num_classes=num_classes).to(device)

In [10]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

In [11]:
#@title Train base model (clean)
input_size = X_train_tensor.shape[1]
num_classes = len(torch.unique(y_train_tensor))

model = TabularNN(input_size=input_size, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  y_test_tensor),  batch_size=256, shuffle=False)

results_file = "results_baseline/url_evaluation_results.txt"
os.makedirs(os.path.dirname(results_file), exist_ok=True)

with open(results_file, "w") as f:
    f.write("Model Evaluation Results\n" + "="*50 + "\n")

epochs = 100
for epoch in range(epochs):
    model.train()
    run = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        run += float(loss.item())

    ep_loss = run / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {ep_loss:.4f}")
    with open(results_file, "a") as f:
        f.write(f"Epoch [{epoch+1}/{epochs}] Loss: {ep_loss:.4f}\n")

# Clean test accuracy
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor.to(device))
    pred = torch.argmax(logits, dim=1).cpu()

clean_acc = accuracy_score(y_test_tensor.cpu().numpy(), pred.numpy())
print(f"Clean test accuracy: {clean_acc:.4f}")

with open(results_file, "a") as f:
    f.write(f"\nClean test accuracy: {clean_acc:.4f}\n")

Epoch [1/100] Loss: 0.6431
Epoch [2/100] Loss: 0.5572
Epoch [3/100] Loss: 0.5382
Epoch [4/100] Loss: 0.5239
Epoch [5/100] Loss: 0.5133
Epoch [6/100] Loss: 0.5087
Epoch [7/100] Loss: 0.5036
Epoch [8/100] Loss: 0.5015
Epoch [9/100] Loss: 0.4984
Epoch [10/100] Loss: 0.4958
Epoch [11/100] Loss: 0.4932
Epoch [12/100] Loss: 0.4900
Epoch [13/100] Loss: 0.4891
Epoch [14/100] Loss: 0.4862
Epoch [15/100] Loss: 0.4845
Epoch [16/100] Loss: 0.4830
Epoch [17/100] Loss: 0.4808
Epoch [18/100] Loss: 0.4795
Epoch [19/100] Loss: 0.4789
Epoch [20/100] Loss: 0.4772
Epoch [21/100] Loss: 0.4772
Epoch [22/100] Loss: 0.4754
Epoch [23/100] Loss: 0.4740
Epoch [24/100] Loss: 0.4729
Epoch [25/100] Loss: 0.4728
Epoch [26/100] Loss: 0.4712
Epoch [27/100] Loss: 0.4705
Epoch [28/100] Loss: 0.4707
Epoch [29/100] Loss: 0.4685
Epoch [30/100] Loss: 0.4687
Epoch [31/100] Loss: 0.4673
Epoch [32/100] Loss: 0.4662
Epoch [33/100] Loss: 0.4654
Epoch [34/100] Loss: 0.4656
Epoch [35/100] Loss: 0.4640
Epoch [36/100] Loss: 0.4638
E

In [12]:
#@title Evaluate base model on adversarial matrices (pre-defense)
def robust_load_matrix(path: str) -> np.ndarray:
    try:
        return np.loadtxt(path, delimiter=",", dtype=np.float32)
    except Exception:
        return np.loadtxt(path, dtype=np.float32)


adversarial_loaders = {
    "PGD": "X_adv_pgd_unsw2_brmulti.txt",
    "JSMA": "X_adv_jsma_unsw_multi.txt",
    "ZOO": "X_adv_zoo_agg_unsw_brmulti.txt",
    "DeepFool": "X_adv_pgd_unsw2_brmulti.txt",
    "FGSM": "X_adv_fgsm_unsw2_brmulti.txt",
    #"CW": "X_adv_cw_agg_unsw.txt",
    #"LPF": "X_adv_lpf_unsw.csv",
    #"HSJ": "X_adv_hsj_unsw.txt",
}

for attack_name, file_path in adversarial_loaders.items():
    X_adv = robust_load_matrix(file_path)

    if X_adv.shape[1] != X_train_tensor.shape[1]:
        print(f"[SKIP] {attack_name}: {X_adv.shape[1]} features != {X_train_tensor.shape[1]}")
        continue

    X_adv_tensor = torch.tensor(X_adv, dtype=torch.float32).to(device)
    t0 = time.time()
    with torch.no_grad():
        adv_pred = model(X_adv_tensor).argmax(1).cpu().numpy()
    dt = time.time() - t0

    adv_acc = accuracy_score(y_test_tensor.numpy(), adv_pred)
    print(f"{attack_name} adversarial accuracy: {adv_acc:.4f} | time {dt:.2f}s")
    with open(results_file, "a") as f:
        f.write(f"{attack_name} adversarial accuracy: {adv_acc:.4f}\n")
        f.write(f"Time taken for {attack_name}: {dt:.2f}s\n")


PGD adversarial accuracy: 0.3428 | time 0.00s
JSMA adversarial accuracy: 0.2299 | time 0.00s
ZOO adversarial accuracy: 0.7418 | time 0.00s
DeepFool adversarial accuracy: 0.3428 | time 0.00s
FGSM adversarial accuracy: 0.3274 | time 0.00s


In [13]:
#@title BARS-CE pipeline
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# from your BARS codebase
from smoothing import Smooth2, Noise
from distribution_transformer import distribution_transformers


class BarsClassifierWrapper(nn.Module):
    """
    Wrapper for BARS:
      - forward(x, noise) returns predicted class indices
      - score(x, noise) returns logits
    """
    def __init__(self, base_model: nn.Module, d: int, num_classes: int, device: torch.device):
        super().__init__()
        self.base = base_model.to(device)
        self.device = device
        self.d = int(d)
        self.num_classes = int(num_classes)

        # Freeze classifier during noise fitting
        self.base.eval()
        for p in self.base.parameters():
            p.requires_grad_(False)

    def forward(self, x: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        logits = self.base(x + noise)
        return torch.argmax(logits, dim=1)

    def score(self, x: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        return self.base(x + noise)


class CEWithBarsRegularizer(nn.Module):
    """
    Cross-entropy + mild regularizer on the noise-generator weights.
    """
    def __init__(self, lambda_reg: float = 1e-3):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(reduction="mean")
        self.lambda_reg = float(lambda_reg)

    def forward(self, logits: torch.Tensor, weight: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = self.ce(logits, targets)
        reg = torch.log1p(torch.exp(-weight.abs())).sum()
        return ce_loss + self.lambda_reg * reg


def optimizing_noise_ce(
    x_train: torch.Tensor,              # [N, D] float
    y_train: torch.Tensor,              # [N] long class indices
    classifier: BarsClassifierWrapper,
    noise_generator: Noise,
    # ----- shape stage -----
    learning_rate_shape: float = 1e-3,
    nt_shape: int = 32,
    num_epochs_shape: int = 300,
    lambda_reg: float = 1e-3,
    batch_size_shape: int = 128,
    print_step: int = 50,
    # ----- scale stage -----
    d: int = None,
    num_classes: int = None,
    n0: int = 100,
    n: int = 1000,
    alpha: float = 1e-3,
    init_step_size_scale: float = 1e-2,
    init_ptb_t_scale: float = 1e-2,
    decay_factor_scale: float = 0.5,
    max_decay_scale: int = 8,
    max_iter_scale: int = 200,
    batch_size_iteration: int = 256,
    batch_size_memory: int = 4096,
    save_dir: str = "bars_fit_ce",
    verbose: bool = True,
):
    """
    1) Fit noise-transform shape using supervised CE + regularizer
    2) Optimize global scale t by hill-climbing mean certified radius
    Returns: (noise_generator, t)
    """
    if d is None:
        d = x_train.shape[1]
    d = int(d)

    if num_classes is None:
        raise ValueError("num_classes must be provided.")

    assert noise_generator.d == d, f"noise_generator.d={noise_generator.d}, expected d={d}"

    dev = classifier.device
    os.makedirs(save_dir, exist_ok=True)

    # ---------------------------
    # SHAPE STAGE
    # ---------------------------
    ds = TensorDataset(x_train.to(dev).float(), y_train.to(dev).long())
    dl = DataLoader(ds, batch_size=batch_size_shape, shuffle=True, drop_last=False)

    crit = CEWithBarsRegularizer(lambda_reg=lambda_reg).to(dev)
    opt = optim.Adam(noise_generator.distribution_transformer.parameters(), lr=learning_rate_shape)

    if verbose:
        print("[BARS-CE] optimizing noise SHAPE")

    for epoch in range(1, num_epochs_shape + 1):
        classifier.base.eval()  # keep classifier frozen
        running_loss, steps = 0.0, 0

        for step_i, (Xb, yb) in enumerate(dl, start=1):
            Xb = Xb.to(dev).float()
            yb = yb.to(dev).long()

            B = Xb.shape[0]
            Xrep = Xb.repeat((nt_shape, 1))
            yrep = yb.repeat_interleave(nt_shape)

            z = noise_generator.sample_norm(nt_shape * B)
            nfeat = noise_generator.norm_to_feat(z)

            logits = classifier.score(Xrep, nfeat)
            loss = crit(logits, noise_generator.get_weight(), yrep)

            opt.zero_grad()
            loss.backward()
            opt.step()

            running_loss += float(loss.item())
            steps += 1

            if verbose and (step_i % print_step == 0):
                print(f"[shape] epoch {epoch}/{num_epochs_shape} "
                      f"step {step_i}/{len(dl)} loss={running_loss / max(1, steps):.4f}")

        if verbose:
            print(f"[shape] epoch {epoch}/{num_epochs_shape} avg_loss={running_loss / max(1, steps):.4f}")

    torch.save(
        noise_generator.distribution_transformer.state_dict(),
        os.path.join(save_dir, "checkpoint-distribution-transformer.state_dict")
    )

    # ---------------------------
    # SCALE STAGE
    # ---------------------------
    if verbose:
        print("[BARS-CE] optimizing global SCALE t")

    # NOTE: adapt constructor signature if your local Smooth2 differs.
    smoother = Smooth2(classifier, d, num_classes, noise_generator, dev)

    Nscale = min(1000, x_train.shape[0])
    sel = torch.randperm(x_train.shape[0])[:Nscale]
    Xs = x_train[sel].to(dev).float()
    ys = y_train[sel].to(dev).long()

    def mean_radius_correct(t_val: float) -> float:
        """
        Average certified radius over correctly certified samples.
        """
        vals = []
        scale_dl = DataLoader(
            TensorDataset(Xs, ys),
            batch_size=batch_size_iteration,
            shuffle=False,
            drop_last=False
        )

        for xb, yb in scale_dl:
            cA_hat, _, R_arr = smoother.bars_certify(
                xb,
                n0=n0,
                n=n,
                t=t_val,
                alpha=alpha,
                batch_size_memory=batch_size_memory
            )

            cA_hat = np.asarray(cA_hat)
            R_arr = np.asarray(R_arr)
            y_np = yb.cpu().numpy().astype(int)

            ok_mask = (cA_hat == y_np) & (cA_hat != smoother.ABSTAIN)
            if ok_mask.any():
                vals.append(float(np.mean(R_arr[ok_mask])))

        return float(np.mean(vals)) if len(vals) else 0.0

    t = 0.0
    ptb = init_ptb_t_scale
    step_size = init_step_size_scale
    decays = 0
    it = 0
    last_sign = 1

    with torch.no_grad():
        while (it < max_iter_scale) and (decays < max_decay_scale):
            base_val = mean_radius_correct(t)
            base_ptb = mean_radius_correct(t + ptb)

            grad_est = (base_ptb - base_val) / (ptb + 1e-12)
            sgn = 1 if grad_est >= 0 else -1

            if sgn != last_sign:
                ptb *= decay_factor_scale
                step_size *= decay_factor_scale
                decays += 1
                last_sign = sgn

            t = max(t + step_size * sgn, 0.0)
            it += 1

            if verbose:
                print(f"[scale] iter {it} | t={t:.6e} | meanR={base_val:.6e} "
                      f"| step={step_size:.6e} | sign={sgn}")

    with open(os.path.join(save_dir, "t"), "w") as f:
        f.write(f"{t:.8e}\n")

    if verbose:
        print(f"[BARS-CE] done. learned t={t:.6f}")

    return noise_generator, t


def train_bars_ce(
    base_model: nn.Module,
    X_train_tensor: torch.Tensor,
    y_train_tensor: torch.Tensor,
    num_classes: int,
    save_dir: str = "bars_fit_ce",
    arch: str = None,
    lambda_reg: float = 1e-3,
    # shape stage
    lr_shape: float = 1e-3,
    nt_shape: int = 32,
    epochs_shape: int = 300,
    # scale stage
    n0: int = 100,
    n: int = 1000,
    alpha: float = 1e-3,
    init_step_t: float = 1e-2,
    init_ptb_t: float = 1e-2,
    decay_factor: float = 0.5,
    max_decay: int = 8,
    max_iter: int = 200,
    batch_iter: int = 256,
    batch_mem: int = 4096,
    print_step: int = 50,
    device: torch.device = None,
):
    """
    High-level convenience wrapper:
      - freezes base model
      - builds noise generator
      - fits noise shape + scale
    Returns: (noise_generator, t)
    """
    if device is None:
        device = next(base_model.parameters()).device

    d = int(X_train_tensor.shape[1])

    if arch is None:
        keys = list(distribution_transformers.keys())
        if len(keys) == 0:
            raise ValueError("distribution_transformers is empty.")
        arch = keys[0]

    if arch not in distribution_transformers:
        raise ValueError(f"Unknown arch '{arch}'. Available: {list(distribution_transformers.keys())}")

    clf = BarsClassifierWrapper(
        base_model=base_model,
        d=d,
        num_classes=num_classes,
        device=device
    ).to(device)

    DT = distribution_transformers[arch](d).to(device)
    noise_gen = Noise(DT, d, device)

    ng, t = optimizing_noise_ce(
        x_train=X_train_tensor.to(device).float(),
        y_train=y_train_tensor.to(device).long(),
        classifier=clf,
        noise_generator=noise_gen,
        learning_rate_shape=lr_shape,
        nt_shape=nt_shape,
        num_epochs_shape=epochs_shape,
        lambda_reg=lambda_reg,
        batch_size_shape=max(batch_iter, 128),
        print_step=print_step,
        d=d,
        num_classes=num_classes,
        n0=n0,
        n=n,
        alpha=alpha,
        init_step_size_scale=init_step_t,
        init_ptb_t_scale=init_ptb_t,
        decay_factor_scale=decay_factor,
        max_decay_scale=max_decay,
        max_iter_scale=max_iter,
        batch_size_iteration=max(batch_iter, 256),
        batch_size_memory=batch_mem,
        save_dir=save_dir,
        verbose=True
    )

    try:
        torch.save(
            DT.state_dict(),
            os.path.join(save_dir, "checkpoint-distribution-transformer.state_dict")
        )
    except Exception as e:
        print("[WARN] could not save DT weights:", e)

    return ng, t


# ---------------------------
# Example usage
# ---------------------------
# Make sure y_train_tensor contains class indices 0..K-1
# and model is already trained.

num_classes = int(max(y_train_tensor.max(), y_test_tensor.max()).item()) + 1

print("Available distribution transformer architectures:", list(distribution_transformers.keys()))


chosen_arch = list(distribution_transformers.keys())[0]

noise_gen, t = train_bars_ce(
    base_model=model,
    X_train_tensor=X_train_tensor,
    y_train_tensor=y_train_tensor,
    num_classes=num_classes,
    save_dir="bars_fit_ce",
    arch=chosen_arch,
    lambda_reg=1e-3,
    lr_shape=1e-3,
    nt_shape=32,
    epochs_shape=300,
    n0=100,
    n=1000,
    alpha=1e-3,
    init_step_t=1e-2,
    init_ptb_t=1e-2,
    decay_factor=0.5,
    max_decay=8,
    max_iter=200,
    batch_iter=256,
    batch_mem=4096,
    print_step=50,
    device=device,
)

print("Finished training BARS-CE noise. Learned t:", t)

Available distribution transformer architectures: ['gaussian', 'isru', 'isru_gaussian', 'isru_gaussian_arctan']
[BARS-CE] optimizing noise SHAPE
[shape] epoch 1/300 step 50/685 loss=57.9471
[shape] epoch 1/300 step 100/685 loss=56.0940
[shape] epoch 1/300 step 150/685 loss=54.9268
[shape] epoch 1/300 step 200/685 loss=53.9375
[shape] epoch 1/300 step 250/685 loss=52.8512
[shape] epoch 1/300 step 300/685 loss=51.9814
[shape] epoch 1/300 step 350/685 loss=51.1888
[shape] epoch 1/300 step 400/685 loss=50.5704
[shape] epoch 1/300 step 450/685 loss=50.1043
[shape] epoch 1/300 step 500/685 loss=49.6777
[shape] epoch 1/300 step 550/685 loss=49.2422
[shape] epoch 1/300 step 600/685 loss=48.8357
[shape] epoch 1/300 step 650/685 loss=48.4779
[shape] epoch 1/300 avg_loss=48.2958
[shape] epoch 2/300 step 50/685 loss=44.2121
[shape] epoch 2/300 step 100/685 loss=44.0750
[shape] epoch 2/300 step 150/685 loss=43.9304
[shape] epoch 2/300 step 200/685 loss=43.6242
[shape] epoch 2/300 step 250/685 loss=

In [14]:
#@title Evaluation of BARS-CE against clean + adversarial matrices

@torch.no_grad()
def _load_adv_matrix(path: str) -> np.ndarray:
    # tries CSV first, else whitespace
    try:
        return np.loadtxt(path, delimiter=",", dtype=np.float32)
    except Exception:
        return np.loadtxt(path, dtype=np.float32)

@torch.no_grad()
def evaluate_bars_on_adversarial_loaders(
    base_model: nn.Module,
    noise_gen: Noise,
    t: float,
    X_test_tensor: torch.Tensor,
    y_test_tensor: torch.Tensor,
    adversarial_loaders: dict,
    num_classes: int = 2,
    n0: int = 200,
    n: int = 1000,
    alpha: float = 0.001,
    batch_size_iter: int = 256,
    batch_size_memory: int = 2048,
    results_file: str = "results_bars_eval.txt"
):
    base_model.eval()

    X_test = X_test_tensor.to(device).float()
    y_test = y_test_tensor.cpu().numpy().astype(int)
    N, D = X_test.shape

    # wrap the same way we did during training
    clf = BarsClassifierWrapper(
        base_model=base_model,
        d=D,
        num_classes=num_classes,
        device=device
    ).to(device)

    smoothed = Smooth2(
        clf,
        d=D,
        num_classes=num_classes,
        noise_generator=noise_gen,
        device=device
    )

    # make sure output directory for results_file exists
    if os.path.dirname(results_file):
        os.makedirs(os.path.dirname(results_file), exist_ok=True)

    with open(results_file, "w") as f:
        f.write("===== BARS-CE Evaluation on Adversarial Sets =====\n")
        f.write(f"t={t:.6f}, n0={n0}, n={n}, alpha={alpha}, "
                f"batch_iter={batch_size_iter}, batch_mem={batch_size_memory}\n\n")

    def _plain_accuracy(X_adv_np: np.ndarray) -> float:
        Xa = torch.tensor(X_adv_np, dtype=torch.float32, device=device)
        logits = base_model(Xa)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        return float((preds == y_test).mean() * 100.0)

    for name, path in adversarial_loaders.items():
        # Load adv matrix or use clean test set
        if name.lower() == "clean":
            X_adv_np = X_test.cpu().numpy().astype(np.float32)
        else:
            X_adv_np = _load_adv_matrix(path).astype(np.float32)

            # sanity check for feature dim
            if X_adv_np.shape[1] != D:
                print(f"[WARN] {name}: got {X_adv_np.shape[1]} dims, expected {D}. Skipping.")
                continue

            # align row count with test set
            if X_adv_np.shape[0] < N:
                print(f"[WARN] {name}: {X_adv_np.shape[0]} rows < test size {N}. Skipping.")
                continue
            if X_adv_np.shape[0] > N:
                X_adv_np = X_adv_np[:N]

        # ----- Plain (non-smoothed) accuracy
        plain_acc = _plain_accuracy(X_adv_np)

        # ----- Certification via Smooth2.bars_certify
        cAs_all = []
        radii_all = []

        X_adv = torch.tensor(X_adv_np, dtype=torch.float32, device=device)
        for i in range(0, N, batch_size_iter):
            xb = X_adv[i:i+batch_size_iter]

            # cA_hat: predicted class per sample OR ABSTAIN
            # radius_feat_dim: per-feature radii (if Smooth2 exposes that)
            # radius_feat: scalar radius per sample
            cA_hat, radius_feat_dim, radius_feat = smoothed.bars_certify(
                xb,
                n0=n0,
                n=n,
                t=t,
                alpha=alpha,
                batch_size_memory=batch_size_memory
            )
            cAs_all.append(cA_hat)
            radii_all.append(radius_feat)

        cAs_all = np.concatenate(cAs_all, axis=0)
        radii_all = np.concatenate(radii_all, axis=0)
        # compute metrics
        not_abstain = (cAs_all != smoothed.ABSTAIN)
        cert_correct = ((cAs_all == y_test) & not_abstain).mean() * 100.0
        abstain_rate = (~not_abstain).mean() * 100.0
        avg_radius = float(radii_all[not_abstain].mean()) if not_abstain.any() else 0.0

        line = (f"[BARS-CE|{name}] "
                f"PlainAcc={plain_acc:.2f}% | "
                f"CertAcc={cert_correct:.2f}% | "
                f"Abstain={abstain_rate:.2f}% | "
                f"AvgR={avg_radius:.4f}")
        print(line)
        with open(results_file, "a") as f:
            f.write(line + "\n")

    print(f"\n[BARS-CE] evaluation complete. Saved to: {results_file}")


In [15]:
evaluate_bars_on_adversarial_loaders(
    base_model=model,
    noise_gen=noise_gen,
    t=t,
    X_test_tensor=X_test_tensor,
    y_test_tensor=y_test_tensor,
    adversarial_loaders=adversarial_loaders,
    num_classes=num_classes,
    n0=200,
    n=1000,
    alpha=0.001,
    batch_size_iter=256,
    batch_size_memory=2048,
    results_file="results_bars_eval.txt"
)

[BARS-CE|PGD] PlainAcc=34.28% | CertAcc=34.11% | Abstain=3.92% | AvgR=0.0699
[BARS-CE|JSMA] PlainAcc=22.99% | CertAcc=13.81% | Abstain=23.63% | AvgR=0.0502
[BARS-CE|ZOO] PlainAcc=74.18% | CertAcc=52.53% | Abstain=14.35% | AvgR=0.0505
[BARS-CE|DeepFool] PlainAcc=34.28% | CertAcc=34.11% | Abstain=3.92% | AvgR=0.0699
[BARS-CE|FGSM] PlainAcc=32.74% | CertAcc=31.34% | Abstain=8.07% | AvgR=0.0564

[BARS-CE] evaluation complete. Saved to: results_bars_eval.txt
